# Lesson 11: GANs — the forger vs the detective 🕵️
**Learning goals**
1. Understand GANs: the **generator** makes fakes, the **discriminator** catches fakes, and both improve together
2. Train a GAN yourself and watch it go from static to handwritten digits
3. Learn the idea behind VAEs, and how GANs relate to deepfakes

**⏱️ Timing:** analogy 10' | build 15' | train & observe 25' | discussion 10'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 💵 An analogy first
- **The forger (generator, G):** at first, their fake banknotes are obviously fake
- **The detective (discriminator, D):** at first, not very good at spotting fakes either
- Every round: the detective studies real and fake notes and learns the difference; the forger improves based on "why I got caught"
- After many rounds, the fakes are so good the detective can only **guess (50%)** — and the generator has won

> Notice: the discriminator is exactly the **discriminative AI** from Lesson 2! A GAN = discriminative + generative AI training each other.

Today our "banknotes" are **handwritten digits** (the MNIST dataset: 70,000 handwritten digits, 28×28 pixels each).

In [ ]:
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
data = datasets.MNIST(".", train=True, download=True,
                      transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])]))
loader = torch.utils.data.DataLoader(data, batch_size=128, shuffle=True)

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(data[i][0][0], cmap="gray"); ax.axis("off")
plt.suptitle("real digits (the 'real money')"); plt.show()

## Build the two AIs
- Generator: takes 64 **random numbers** (like dice) and outputs a 28×28 picture
- Discriminator: takes a picture and outputs "the probability it's real"

In [ ]:
NOISE = 64

G = nn.Sequential(                       # the forger
    nn.Linear(NOISE, 256), nn.ReLU(),
    nn.Linear(256, 512), nn.ReLU(),
    nn.Linear(512, 784), nn.Tanh()).to(device)

D = nn.Sequential(                       # the detective
    nn.Linear(784, 512), nn.LeakyReLU(0.2),
    nn.Linear(512, 256), nn.LeakyReLU(0.2),
    nn.Linear(256, 1), nn.Sigmoid()).to(device)

loss_fn = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4)
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4)
fixed_noise = torch.randn(16, NOISE, device=device)   # fixed "dice" so we can watch progress

def show_fakes(title):
    with torch.no_grad():
        fakes = G(fixed_noise).view(-1, 28, 28).cpu()
    fig, axes = plt.subplots(2, 8, figsize=(10, 2.6))
    for ax, f in zip(axes.flat, fakes):
        ax.imshow(f, cmap="gray"); ax.axis("off")
    plt.suptitle(title); plt.show()

show_fakes("before training: pure garbage")

## Let the contest begin! 🥊
Each round (epoch) goes through all 60,000 pictures. Watch two numbers:
- `D(real)`: how sure the detective is that real pictures are real (should be close to 1)
- `D(fake)`: how sure the detective is that fake pictures are real (the forger wants it high, the detective wants it low)

In [ ]:
EPOCHS = 10   # about 10 seconds per epoch on a GPU; much slower on CPU

for epoch in range(1, EPOCHS + 1):
    for real, _ in loader:
        real = real.view(-1, 784).to(device)
        n = real.size(0)
        fake = G(torch.randn(n, NOISE, device=device))

        # 1) Train the detective: say 1 for real, 0 for fake
        d_real, d_fake = D(real), D(fake.detach())
        loss_D = loss_fn(d_real, torch.ones_like(d_real)) + loss_fn(d_fake, torch.zeros_like(d_fake))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # 2) Train the forger: try to make the detective say 1 for fakes
        d_fake = D(fake)
        loss_G = loss_fn(d_fake, torch.ones_like(d_fake))
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

    print(f"epoch {epoch}  D(real)={d_real.mean():.2f}  D(fake)={d_fake.mean():.2f}")
    if epoch in (1, 3, 5, EPOCHS):
        show_fakes(f"after epoch {epoch}")

### 🧩 Exercise 1: Observation log
| Epoch | The pictures look like… | D(fake) |
|---|---|---|
| 0 | | |
| 1 | | |
| 5 | | |
| 10 | | |

**Discuss:** why doesn't `D(fake)` just keep falling, but wobble around 0.3 instead? (Hint: both players are improving **at the same time**.)

### 🧩 Exercise 2: Same forger, different dice
Different random numbers in → a different digit out. Run the cell below a few times.

In [ ]:
with torch.no_grad():
    f = G(torch.randn(1, NOISE, device=device)).view(28, 28).cpu()
plt.imshow(f, cmap="gray"); plt.axis("off"); plt.show()

### 🚀 Challenge
1. **Morphing between two digits:** take two sets of random numbers a and b and draw 8 pictures from a → b (code below). How does the digit gradually change shape?
2. Change `lr` to `2e-3` (10× bigger) and retrain. What goes wrong? (You might see "mode collapse": it only draws one digit.)

In [ ]:
a, b = torch.randn(NOISE, device=device), torch.randn(NOISE, device=device)
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
with torch.no_grad():
    for ax, t in zip(axes, torch.linspace(0, 1, 8)):
        ax.imshow(G((1 - t) * a + t * b).view(28, 28).cpu(), cmap="gray"); ax.axis("off")
plt.show()

## 🔗 Connecting to Lesson 9
The 64 random numbers the generator takes in are a **latent space**, just like the autoencoder's code in Lesson 9. The morphing challenge above is a walk through that latent space.
| Model | How it learns to create |
|---|---|
| **VAE** (Lesson 9) | Squeeze into a latent code, then rebuild |
| **Diffusion** (Lesson 10) | Learn to remove noise, step by step |
| **GAN** (this lesson) | A forger and a detective compete |

## ⚠️ GANs and deepfakes
GANs can create faces that look completely real (none of the people on the website *thispersondoesnotexist* exist). **Discuss:**
- What are good uses of this technology? (game characters, film effects, anonymous avatars that protect privacy…)
- What are the dangers? (impersonating people, fake news, scams…) We'll discuss this properly in Lesson 15.

## 📌 Summary
- GAN = generator (forger) + discriminator (detective), competing and improving together
- Random numbers are the "seed": different numbers → different creations
- VAE = compress + decompress; diffusion = add noise + remove noise; GAN = compete — three different ways to "create"